# Kenya Maize Advisory Chatbot

## Local PDF RAG for English & Swahili

Answers questions about **maize farming in Kenya** by retrieving evidence from the documents in `pdfs/` plus the project's own `DISEASE_INFO` seed knowledge, then generating a grounded, source-cited reply.

```
PDFs --> extract --> clean --> language tag --> chunk
      --> (section 3) embed --> hybrid retrieval --> guardrails --> generate --> translate --> cite
        DISEASE_INFO (4 leaf diseases) feeds in at the chunking step
```

- **Languages:** English and Swahili, using a translation pivot (Swahili query -> English retrieval/generation -> Swahili answer).
- **Stack:** everything free and local - `pymupdf`, `multilingual-e5-small`, FAISS/numpy, `rank_bm25`, opus-mt, Qwen2.5. No API keys.
- **Artifacts:** `rag_data/` (corpus, chunks), `rag_artifacts/` (index, config, evaluation report), MySQL chat history (section 11).

Sections 1-2 below build the environment and the corpus. Later sections embed, retrieve, evaluate, tune and deploy.

## 1. Environment & Setup

### 1.1 Project objectives

- Build a knowledge base from the five PDFs in `pdfs/` plus `DISEASE_INFO`, covering Kenyan maize farming: agronomy, production statistics, farm economics and the four leaf diseases.
- Retrieve answers to farmer questions in **English or Swahili** and generate replies that stay inside the retrieved evidence, with citations back to source page numbers.
- Export a reusable pipeline (`rag_pipeline.py`) that the Streamlit app loads as a **Maize Advisory** chat tab, continuing the conversation after an image classification.

Everything runs on the CPU, locally, for free.

### 1.2 Dependency installation

`pymupdf`, `pandas` and `torch` are already available in this environment. The heavier RAG packages (`sentence-transformers`, `rank-bm25`, `faiss-cpu`) are only needed from section 3 onwards.

Set `INSTALL_DEPENDENCIES = True` once if the check below reports them missing, run the cell, then **restart the kernel** and set the flag back to `False`.

In [1]:
INSTALL_DEPENDENCIES = False  # set True once, restart the kernel afterwards

if INSTALL_DEPENDENCIES:
    %pip install -q sentence-transformers rank-bm25 faiss-cpu pymupdf

import importlib.util
import pandas as pd

REQUIRED = {
    "pymupdf": "PDF text extraction (section 2)",
    "pandas": "audit tables (section 2)",
    "torch": "embedding model runtime (section 3)",
    "sentence_transformers": "embeddings (section 3)",
    "rank_bm25": "hybrid search (section 4)",
    "faiss": "vector index (section 3, numpy fallback available)",
}
availability = [
    {"package": name, "installed": importlib.util.find_spec(name) is not None, "needed for": purpose}
    for name, purpose in REQUIRED.items()
]
print(pd.DataFrame(availability).to_string(index=False))
missing = [name for name, row in zip(REQUIRED, availability) if not row["installed"]]
blocking_now = [name for name in missing if name in ("pymupdf", "pandas")]
if blocking_now:
    raise ImportError("Missing required packages: " + ", ".join(blocking_now) + ". Set INSTALL_DEPENDENCIES=True and rerun.")
if missing:
    print("\nNot installed yet (needed from section 3):", ", ".join(missing))

              package  installed                                         needed for
              pymupdf       True                    PDF text extraction (section 2)
               pandas       True                           audit tables (section 2)
                torch       True                embedding model runtime (section 3)
sentence_transformers       True                             embeddings (section 3)
            rank_bm25       True                          hybrid search (section 4)
                faiss       True vector index (section 3, numpy fallback available)


### 1.3 Imports, device & seeding

In [2]:
import ast
import hashlib
import json
import random
import re
import time
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pymupdf
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    DEVICE = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"python stack ready | device={DEVICE} | seed={SEED}")
print(f"pymupdf {pymupdf.__version__} | numpy {np.__version__} | pandas {pd.__version__} | torch {torch.__version__}")

python stack ready | device=mps | seed=42
pymupdf 1.28.2 | numpy 2.5.3 | pandas 3.0.6 | torch 2.14.0


### 1.4 RAG configuration

One config block, used by every later section. Chunk size and overlap are starting values - section 7 tunes them against retrieval metrics.

In [3]:
ROOT         = Path.cwd()                 # run the notebook from the repository root
PDF_DIR      = ROOT / "pdfs"
DATA_DIR     = ROOT / "rag_data"
ARTIFACT_DIR = ROOT / "rag_artifacts"

EMBEDDING_MODEL = "intfloat/multilingual-e5-small"   # prefixes: "query: " / "passage: "
LANGUAGES       = ("en", "sw")
CHUNK_WORDS     = 220
OVERLAP_WORDS   = 40
MIN_CHUNK_WORDS = 30
TOP_K           = 5

if not (ROOT / "app.py").is_file() or not PDF_DIR.is_dir():
    raise FileNotFoundError(
        f"Run this notebook from the repository root (expected app.py and pdfs/ under {ROOT})."
    )
DATA_DIR.mkdir(exist_ok=True)
ARTIFACT_DIR.mkdir(exist_ok=True)

print("RAG configuration")
print(f"  root            : {ROOT}")
print(f"  pdfs            : {PDF_DIR} ({len(list(PDF_DIR.glob('*.pdf')))} PDFs)")
print(f"  data dir        : {DATA_DIR}")
print(f"  embedding model : {EMBEDDING_MODEL}  (section 3)")
print(f"  chunking        : {CHUNK_WORDS} words, {OVERLAP_WORDS} overlap, min {MIN_CHUNK_WORDS}")
print(f"  top_k           : {TOP_K}")
print(f"  languages       : {', '.join(LANGUAGES)}")

RAG configuration
  root            : /Users/willard/school/maize_disease
  pdfs            : /Users/willard/school/maize_disease/pdfs (5 PDFs)
  data dir        : /Users/willard/school/maize_disease/rag_data
  embedding model : intfloat/multilingual-e5-small  (section 3)
  chunking        : 220 words, 40 overlap, min 30
  top_k           : 5
  languages       : en, sw


### 1.5 Reuse existing assets (`DISEASE_INFO`)

The Streamlit app holds curated disease knowledge in a `DISEASE_INFO` dict. The four disease entries become seed documents so the chatbot can answer symptom and management questions even before a PDF covers them.

`app.py` executes Streamlit code at import time, so the dict is read with the `ast` module instead of importing the file.

In [4]:
app_source = (ROOT / "app.py").read_text(encoding="utf-8")
tree = ast.parse(app_source)

DISEASE_INFO = None
for node in tree.body:
    target = node.target.id if isinstance(node, ast.AnnAssign) and isinstance(node.target, ast.Name) else None
    if target is None and isinstance(node, ast.Assign):
        target = node.targets[0].id if isinstance(node.targets[0], ast.Name) else None
    if target == "DISEASE_INFO":
        DISEASE_INFO = ast.literal_eval(node.value)
        break

if DISEASE_INFO is None:
    raise ValueError("DISEASE_INFO not found in app.py")

EXPECTED_CLASSES = ["Blight", "Common_Rust", "Gray_Leaf_Spot", "Healthy"]
if set(DISEASE_INFO) != set(EXPECTED_CLASSES):
    raise ValueError(f"Unexpected classes in DISEASE_INFO: {sorted(DISEASE_INFO)}")

for cls, info in DISEASE_INFO.items():
    missing_fields = {"pathogen", "symptoms", "favourable", "management"} - set(info)
    if missing_fields:
        raise ValueError(f"{cls} is missing fields: {sorted(missing_fields)}")

print(f"loaded DISEASE_INFO: {len(DISEASE_INFO)} diseases x {len(next(iter(DISEASE_INFO.values())))} fields")
for cls, info in DISEASE_INFO.items():
    print(f"  {cls:15s} {info['pathogen']}")

loaded DISEASE_INFO: 4 diseases x 4 fields
  Blight          Exserohilum turcicum (formerly Bipolaris turcica)
  Common_Rust     Puccinia sorghi
  Gray_Leaf_Spot  Cercospora zeae-maydis
  Healthy         -


## 2. Data Ingestion & Chunking

```
pdfs/*.pdf --> 2.1 manifest --> 2.2 page extraction --> 2.3 cleaning --> 2.4 language tag
DISEASE_INFO -------------------------------> 2.5 seed documents --> 2.6 chunking
                                                          --> 2.7 quality check --> 2.8 chunks.jsonl
```

### 2.1 Source manifest & licences

Roles below are **inferred from filenames** - read each PDF and correct the `title` / `topic` fields if the content does not match. `licence` is deliberately `unverified`: check each document's licence before redistributing the corpus, and record what you find in this table (it is saved into `source_manifest.json` in section 2.8).

| source_id | PDF | Inferred role | Topic |
|---|---|---|---|
| S01 | diversity in maize production environments and practices | production environments & on-farm practices | agronomy |
| S02 | MAIZE GROWERS GUIDE | practical growers handbook | agronomy |
| S03 | National-Agriculture-Production-Report-2025 | national production statistics | statistics |
| S04 | Olwande_Smallholder_Maize_Efficiency_Kenya | smallholder maize efficiency study | economics |
| S05 | Planting_strategies_of_maize_farmers_in_Kenya_a_si | research article on planting strategies | agronomy |
| S00 | `DISEASE_INFO` from `app.py` | leaf-disease seed knowledge | disease |

In [5]:
SOURCES = [
    {"source_id": "S00", "filename": None, "title": "Maize leaf disease seed knowledge (app.py)",
     "topic": "disease", "language": "en", "licence": "project asset"},
    {"source_id": "S01", "filename": "diversity in maize production environments and practices.pdf",
     "title": "Diversity in maize production environments and practices", "topic": "agronomy",
     "language": "unknown", "licence": "unverified"},
    {"source_id": "S02", "filename": "MAIZE GROWERS GUIDE.pdf",
     "title": "Maize Growers Guide", "topic": "agronomy", "language": "unknown", "licence": "unverified"},
    {"source_id": "S03", "filename": "National-Agriculture-Production-Report-2025.pdf",
     "title": "National Agriculture Production Report 2025", "topic": "statistics",
     "language": "unknown", "licence": "unverified"},
    {"source_id": "S04", "filename": "Olwande_Smallholder_Maize_Efficiency_Kenya.pdf",
     "title": "Smallholder maize efficiency in Kenya (Olwande)", "topic": "economics",
     "language": "unknown", "licence": "unverified"},
    {"source_id": "S05", "filename": "Planting_strategies_of_maize_farmers_in_Kenya_a_si.pdf",
     "title": "Planting strategies of maize farmers in Kenya", "topic": "agronomy",
     "language": "unknown", "licence": "unverified"},
]

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

PDF_SOURCES = [s for s in SOURCES if s["filename"]]
missing = [s["filename"] for s in PDF_SOURCES if not (PDF_DIR / s["filename"]).is_file()]
if missing:
    raise FileNotFoundError("Put these PDFs in " + str(PDF_DIR) + ":\n" + "\n".join(missing))

SOURCE_HASHES = {}
inventory = []
for src in PDF_SOURCES:
    path = PDF_DIR / src["filename"]
    SOURCE_HASHES[src["source_id"]] = sha256_file(path)
    inventory.append({
        "source_id": src["source_id"], "filename": src["filename"], "topic": src["topic"],
        "size_mb": round(path.stat().st_size / 1e6, 2), "sha256": SOURCE_HASHES[src["source_id"]][:12],
        "licence": src["licence"],
    })

hashes = list(SOURCE_HASHES.values())
if len(set(hashes)) != len(hashes):
    raise ValueError("Two PDFs have identical content - remove the duplicate before chunking.")

display(pd.DataFrame(inventory))
print(f"{len(PDF_SOURCES)} PDFs inventoried, all unique, ready to extract.")

,source_id,filename,topic,size_mb,sha256,licence
0,S01,diversity in maize production environments and...,agronomy,0.67,1aabed3d8f7f,unverified
1,S02,MAIZE GROWERS GUIDE.pdf,agronomy,0.67,454c95229128,unverified
2,S03,National-Agriculture-Production-Report-2025.pdf,statistics,7.58,72a8a2f73d12,unverified
3,S04,Olwande_Smallholder_Maize_Efficiency_Kenya.pdf,economics,0.47,46d25e01d128,unverified
4,S05,Planting_strategies_of_maize_farmers_in_Kenya_...,agronomy,1.08,af7816c7cbfc,unverified


5 PDFs inventoried, all unique, ready to extract.


### 2.2 PDF text extraction

Text is extracted **page by page** and page numbers are kept - they are the citation unit all the way to the chat answer. PyMuPDF does not OCR: a scanned page comes back empty and is flagged as a *review candidate* in the audit. Layout such as tables degrades to running text; check important numbers against the original PDF.

In [6]:
def extract_pages(sources):
    pages = []
    for src in sources:
        path = PDF_DIR / src["filename"]
        doc = pymupdf.open(path)
        if doc.needs_pass:
            raise ValueError(f"{src['filename']} is encrypted - decrypt it first.")
        for page in doc:
            text = page.get_text("text") or ""
            pages.append({
                "source_id": src["source_id"],
                "filename": src["filename"],
                "title": src["title"],
                "topic": src["topic"],
                "page": page.number + 1,
                "n_words": len(text.split()),
                "text": text,
            })
        doc.close()
    return pages

t0 = time.perf_counter()
pdf_pages = extract_pages(PDF_SOURCES)
elapsed = time.perf_counter() - t0

audit = (
    pd.DataFrame(pdf_pages)
    .groupby(["source_id", "filename"], as_index=False)
    .agg(pages=("page", "count"), words=("n_words", "sum"),
         short_pages=("n_words", lambda s: int((s < 50).sum())))
)
display(audit)

shortest = pd.DataFrame(pdf_pages).nsmallest(8, "n_words")[["source_id", "page", "n_words"]]
print("Review candidates (thinnest pages - check they are not failed extractions):")
display(shortest)
print(f"extracted {len(pdf_pages)} pages / {sum(p['n_words'] for p in pdf_pages):,} words in {elapsed:.1f}s")

,source_id,filename,pages,words,short_pages
0,S01,diversity in maize production environments and...,36,1519,17
1,S02,MAIZE GROWERS GUIDE.pdf,19,3385,1
2,S03,National-Agriculture-Production-Report-2025.pdf,202,41719,56
3,S04,Olwande_Smallholder_Maize_Efficiency_Kenya.pdf,28,1743,11
4,S05,Planting_strategies_of_maize_farmers_in_Kenya_...,13,8372,0


Review candidates (thinnest pages - check they are not failed extractions):


,source_id,page,n_words
29,S01,30,0
30,S01,31,0
31,S01,32,0
32,S01,33,0
33,S01,34,0
34,S01,35,0
35,S01,36,0
28,S01,29,2


extracted 298 pages / 56,738 words in 0.7s


### 2.3 Text cleaning

- Unicode normalised to NFC, null bytes and non-breaking spaces removed.
- Hyphenated line breaks rejoined (`inter-\nnational` -> `international`).
- All whitespace collapsed to single spaces - chunking works on word windows, so line and paragraph breaks carry no information downstream.

Cleaning is deliberately conservative: no stopword removal, no stemming, no lowercasing - that is the embedding model's job.

In [7]:
def clean_text(text):
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\x00", " ").replace("\xa0", " ")
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)     # de-hyphenate line breaks
    text = re.sub(r"\s+", " ", text)
    return text.strip()

before = sum(len(p["text"]) for p in pdf_pages)
for page in pdf_pages:
    page["text"] = clean_text(page["text"])
    page["n_words"] = len(page["text"].split())
after = sum(len(p["text"]) for p in pdf_pages)

empty_pages = [p for p in pdf_pages if not p["text"]]
print(f"cleaned {before:,} -> {after:,} chars ({before - after:,} removed)")
print(f"empty pages after cleaning: {len(empty_pages)}")
if empty_pages:
    display(pd.DataFrame(empty_pages)[["source_id", "page", "n_words"]])

cleaned 394,340 -> 371,333 chars (23,007 removed)
empty pages after cleaning: 7


,source_id,page,n_words
0,S01,30,0
1,S01,31,0
2,S01,32,0
3,S01,33,0
4,S01,34,0
5,S01,35,0
6,S01,36,0


### 2.4 Language tagging

A stopword-ratio heuristic tags each page `en`, `sw` or `unknown`. It is deliberately simple and **not** a language detector of record - it only feeds corpus statistics and the routing sanity checks in section 4. `unknown` pages are treated as English (the corpus is predominantly English); `langdetect` or `fasttext lid.176` can replace this cell without changing downstream code.

In [8]:
EN_STOP = {
    "the", "and", "of", "to", "in", "a", "is", "that", "for", "on", "with", "as", "by",
    "at", "from", "this", "are", "be", "it", "or", "an", "was", "were", "which", "has",
    "have", "not", "but", "their", "its", "will", "can", "one", "than", "also", "into",
}
SW_STOP = {
    "ya", "na", "wa", "kwa", "cha", "hii", "hiyo", "hizo", "wao", "yake", "wake",
    "katika", "kuhusu", "ambayo", "ambao", "hivyo", "lakini", "sana", "aidha", "bali",
    "kabla", "baada", "hadi", "pia", "kila", "wakulima", "shamba", "kilimo", "mahindi",
    "mazao", "udongo", "mbolea", "wadudu", "ugonjwa", "magonjwa", "mbegu", "nchini",
    "kutumia", "kupitia", "zaidi", "chini", "juu", "wakati", "hali", "hewa",
}

def detect_language(text):
    tokens = [t for t in re.findall(r"[a-zA-Z]+", text.lower()) if len(t) > 1]
    if not tokens:
        return "unknown", 0, 0
    en_hits = sum(t in EN_STOP for t in tokens)
    sw_hits = sum(t in SW_STOP for t in tokens)
    if sw_hits >= 5 and sw_hits > en_hits:
        return "sw", sw_hits, en_hits
    if en_hits >= 5:
        return "en", en_hits, sw_hits
    return "unknown", en_hits, sw_hits

for page in pdf_pages:
    page["language"], page["en_hits"], page["sw_hits"] = detect_language(page["text"])

page_lang = pd.DataFrame(pdf_pages).groupby(["source_id", "language"], as_index=False).agg(pages=("page", "count"))
display(page_lang.pivot(index="source_id", columns="language", values="pages").fillna(0).astype(int))

for src in PDF_SOURCES:
    pages = [p for p in pdf_pages if p["source_id"] == src["source_id"]]
    votes = [p["language"] for p in pages]
    doc_lang = max(set(votes), key=votes.count)
    src["language"] = "en" if doc_lang == "unknown" else doc_lang
    print(f"{src['source_id']}  {doc_lang:7s} (majority of {len(pages)} pages)  {src['title'][:50]}")

language,en,unknown
source_id,,
S01,9,27
S02,17,2
S03,111,91
S04,23,5
S05,13,0


S01  unknown (majority of 36 pages)  Diversity in maize production environments and pra
S02  en      (majority of 19 pages)  Maize Growers Guide
S03  en      (majority of 202 pages)  National Agriculture Production Report 2025
S04  en      (majority of 28 pages)  Smallholder maize efficiency in Kenya (Olwande)
S05  en      (majority of 13 pages)  Planting strategies of maize farmers in Kenya


### 2.5 Seed conversion (`DISEASE_INFO`)

Each disease becomes one compact document combining pathogen, symptoms, favourable conditions and management advice. Seed pages use `source_id` `S00` and `page` 0 (no PDF page exists), so citations read *"DISEASE_INFO"* instead of a page number.

In [9]:
seed_pages = []
for cls, info in DISEASE_INFO.items():
    parts = [f"Maize leaf disease: {cls.replace('_', ' ')}. Pathogen: {info['pathogen']}."]
    if info["pathogen"] != "-":
        parts.append(f"Symptoms: {info['symptoms']}")
    parts.append(f"Favourable conditions: {info['favourable']}")
    parts.append(f"Management: {info['management']}")
    seed_pages.append({
        "source_id": "S00",
        "filename": None,
        "title": f"Maize leaf disease: {cls}",
        "topic": "disease",
        "page": 0,
        "n_words": len(" ".join(parts).split()),
        "text": clean_text(" ".join(parts)),
        "language": "en",
        "disease": cls,
    })

print(f"{len(seed_pages)} seed documents created:")
for page in seed_pages:
    print(f"  {page['title']:35s} {page['n_words']:3d} words")

4 seed documents created:
  Maize leaf disease: Blight          108 words
  Maize leaf disease: Common_Rust      97 words
  Maize leaf disease: Gray_Leaf_Spot  103 words
  Maize leaf disease: Healthy          43 words


### 2.6 Chunking

Chunks are **overlapping word windows within a page** (default 220 words, 40 overlap). Page boundaries are never crossed, so every chunk carries an exact citation (`source_id` + `page`). A window that adds nothing beyond its predecessor is skipped, exact duplicate texts are dropped, and tiny trailing windows below `MIN_CHUNK_WORDS` are discarded unless they are the page's only chunk.

Metadata on every chunk: `chunk_id, source_id, filename, title, topic, language, page`, plus `disease` for seed chunks.

In [10]:
def make_chunks(pages, chunk_words=CHUNK_WORDS, overlap_words=OVERLAP_WORDS, min_words=MIN_CHUNK_WORDS):
    if chunk_words < 1 or not 0 <= overlap_words < chunk_words:
        raise ValueError("Require chunk_words > 1 and 0 <= overlap_words < chunk_words.")
    step = chunk_words - overlap_words
    chunks, duplicates, seen = [], [], {}
    for page in pages:
        words = page["text"].split()
        if not words:
            continue
        prev_end, window_index = 0, 0
        for start in range(0, len(words), step):
            window = words[start:start + chunk_words]
            if start + len(window) <= prev_end:
                continue                      # fully covered by the previous window
            if len(window) < min_words and start > 0 and len(words) > min_words:
                continue                      # tiny tail, already mostly covered
            prev_end = start + len(window)
            window_index += 1
            text = " ".join(window)
            text_hash = hashlib.sha256(text.encode("utf-8")).hexdigest()
            chunk_id = f"{page['source_id']}-p{page['page']:04d}-c{window_index:03d}"
            if text_hash in seen:
                duplicates.append({"removed": chunk_id, "kept": seen[text_hash]})
                continue
            seen[text_hash] = chunk_id
            chunks.append({
                "chunk_id": chunk_id,
                "source_id": page["source_id"],
                "filename": page.get("filename"),
                "title": page["title"],
                "topic": page["topic"],
                "language": page["language"],
                "page": page["page"],
                "disease": page.get("disease"),
                "n_words": len(window),
                "text_sha256": text_hash,
                "text": text,
            })
    return chunks, duplicates

all_pages = seed_pages + pdf_pages
chunks, duplicate_chunks = make_chunks(all_pages)
print(f"{len(all_pages)} pages -> {len(chunks)} chunks "
      f"({len(duplicate_chunks)} exact duplicates removed)")

302 pages -> 458 chunks (1 exact duplicates removed)


### 2.7 Chunk quality inspection

Check three things before persisting: chunk counts per source, the length distribution (very short chunks retrieve poorly), and a sample of actual text from both a seed document and a PDF.

In [11]:
frame = pd.DataFrame(chunks)
stats = frame.groupby(["source_id", "title"], as_index=False).agg(
    chunks=("chunk_id", "count"), pages=("page", "nunique"),
    words=("n_words", "sum"), mean_words=("n_words", "mean"),
)
stats["mean_words"] = stats["mean_words"].round(0).astype(int)
display(stats)

lengths = frame["n_words"]
print(f"chunk length: min={lengths.min()}  median={int(lengths.median())}  max={lengths.max()}")
print(f"chunks below {MIN_CHUNK_WORDS} words: {(lengths < MIN_CHUNK_WORDS).sum()}")
print("language distribution:", frame["language"].value_counts().to_dict())
print("topic distribution   :", frame["topic"].value_counts().to_dict())

print("\nFive shortest chunks (review their usefulness):")
display(frame.nsmallest(5, "n_words")[["chunk_id", "topic", "n_words", "page"]])

print("\nSample seed chunk (disease):")
disease_chunk = frame[frame["source_id"] == "S00"].iloc[0]
print(f"[{disease_chunk['chunk_id']}] {disease_chunk['text'][:400]}...")

print("\nSample PDF chunk:")
pdf_chunk = frame[frame["source_id"] != "S00"].iloc[0]
print(f"[{pdf_chunk['chunk_id']}] {pdf_chunk['text'][:400]}...")

,source_id,title,chunks,pages,words,mean_words
0,S00,Maize leaf disease: Blight,1,1,108,108
1,S00,Maize leaf disease: Common_Rust,1,1,97,97
2,S00,Maize leaf disease: Gray_Leaf_Spot,1,1,103,103
3,S00,Maize leaf disease: Healthy,1,1,43,43
4,S01,Diversity in maize production environments and...,29,29,1519,52
5,S02,Maize Growers Guide,24,19,3583,149
6,S03,National Agriculture Production Report 2025,326,202,46415,142
7,S04,Smallholder maize efficiency in Kenya (Olwande),27,27,1741,64
8,S05,Planting strategies of maize farmers in Kenya,48,13,9772,204


chunk length: min=2  median=160  max=220
chunks below 30 words: 62
language distribution: {'en': 320, 'unknown': 138}
topic distribution   : {'statistics': 326, 'agronomy': 101, 'economics': 27, 'disease': 4}

Five shortest chunks (review their usefulness):


,chunk_id,topic,n_words,page
32,S01-p0029-c001,agronomy,2,29
387,S04-p0005-c001,economics,4,5
10,S01-p0007-c001,agronomy,5,7
21,S01-p0018-c001,agronomy,5,18
58,S03-p0002-c001,statistics,6,2



Sample seed chunk (disease):
[S00-p0000-c001] Maize leaf disease: Blight. Pathogen: Exserohilum turcicum (formerly Bipolaris turcica). Symptoms: Long, cigar-shaped grey-green to tan lesions running parallel to the veins. Lesions start on lower leaves and move upward. Under favourable conditions they coalesce and the whole leaf dies, giving the crop a burnt appearance. Favourable conditions: Cool to moderate temperatures (18-27 C) with high hu...

Sample PDF chunk:
[S01-p0001-c001] Diversity in Maize Production Environments and Practices in Kenya Introduction and Conference Overview Mary W. K. Mathenge Tegemeo Institute – Egerton University 1...


### 2.8 Persist

Three files land in `rag_data/`:

| File | Contents |
|---|---|
| `chunks.jsonl` | one chunk per line - the unit that gets embedded in section 3 |
| `pages.jsonl` | cleaned page texts - lets section 7 re-chunk with new settings without re-extracting the PDFs |
| `source_manifest.json` | provenance: hashes, page/word counts, languages, config used |

A round-trip read verifies the files are valid before moving on.

In [12]:
DATA_DIR.mkdir(exist_ok=True)

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")

chunks_path = DATA_DIR / "chunks.jsonl"
pages_path = DATA_DIR / "pages.jsonl"
manifest_path = DATA_DIR / "source_manifest.json"
audit_path = DATA_DIR / "page_audit.csv"

write_jsonl(chunks_path, chunks)
write_jsonl(pages_path, all_pages)

manifest = {
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "config": {
        "embedding_model": EMBEDDING_MODEL,
        "chunk_words": CHUNK_WORDS,
        "overlap_words": OVERLAP_WORDS,
        "languages": list(LANGUAGES),
        "seed": SEED,
    },
    "sources": [
        {
            "source_id": src["source_id"],
            "filename": src["filename"],
            "title": src["title"],
            "topic": src["topic"],
            "language": src["language"],
            "licence": src["licence"],
            "sha256": SOURCE_HASHES.get(src["source_id"]),
            "chunks": sum(1 for c in chunks if c["source_id"] == src["source_id"]),
        }
        for src in SOURCES
    ],
}
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
pd.DataFrame(pdf_pages).drop(columns=["text"]).to_csv(audit_path, index=False)

# round-trip verification
with open(chunks_path, encoding="utf-8") as handle:
    reloaded = [json.loads(line) for line in handle]
assert len(reloaded) == len(chunks), "chunks.jsonl round-trip failed"
assert reloaded[0]["chunk_id"] == chunks[0]["chunk_id"]
assert len(list(open(pages_path, encoding="utf-8"))) == len(all_pages)

for path in (chunks_path, pages_path, manifest_path, audit_path):
    print(f"{path.relative_to(ROOT)}  {path.stat().st_size / 1024:.1f} KB")
print(f"\n{len(chunks)} chunks persisted - section 3 embeds them.")

rag_data/chunks.jsonl  564.5 KB
rag_data/pages.jsonl  439.8 KB
rag_data/source_manifest.json  2.2 KB
rag_data/page_audit.csv  36.0 KB

458 chunks persisted - section 3 embeds them.


## 3. Embeddings & Vector Store
An **embedding** is just a list of numbers - 384 of them - that captures *meaning*. Texts with similar meanings end up close together, even in different languages.

### 3.1 Load the embedding model

In [13]:
from sentence_transformers import SentenceTransformer

t0 = time.perf_counter()
embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
print(f"loaded {EMBEDDING_MODEL} on {DEVICE} in {time.perf_counter() - t0:.1f}s")
print(f"max sequence length: {embedder.max_seq_length} tokens  (chunks are ~220 words, safely below)")

/opt/homebrew/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7178.69it/s]

loaded intfloat/multilingual-e5-small on mps in 10.3s
max sequence length: 512 tokens  (chunks are ~220 words, safely below)


### 3.2 Embed the corpus

One call, one vector per chunk, **in the same order as `chunks`** - vector row 500 always belongs to chunk 500. That row-number rule is the only thing you need to remember for the whole index.

In [14]:
def embed_texts(texts, kind):
    """kind is "query" (questions) or "passage" (documents). e5 needs the prefix."""
    prefix = "query: " if kind == "query" else "passage: "
    vectors = embedder.encode(
        [prefix + text for text in texts],
        batch_size=32,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    return np.asarray(vectors, dtype="float32")

t0 = time.perf_counter()
chunk_vectors = embed_texts([chunk["text"] for chunk in chunks], "passage")
print(f"embedded {chunk_vectors.shape[0]} chunks -> {chunk_vectors.shape[1]} dimensions "
      f"in {time.perf_counter() - t0:.1f}s")


Batches:   0%|          | 0/15 [00:00<?, ?it/s]


Batches:   7%|▋         | 1/15 [00:00<00:13,  1.01it/s]


Batches:  13%|█▎        | 2/15 [00:02<00:15,  1.21s/it]


Batches:  20%|██        | 3/15 [00:03<00:16,  1.37s/it]


Batches:  27%|██▋       | 4/15 [00:05<00:16,  1.51s/it]

...


Batches:  80%|████████  | 12/15 [00:13<00:02,  1.36it/s]


Batches:  87%|████████▋ | 13/15 [00:13<00:01,  1.76it/s]


Batches: 100%|██████████| 15/15 [00:13<00:00,  3.02it/s]


Batches: 100%|██████████| 15/15 [00:13<00:00,  1.07it/s]

embedded 458 chunks -> 384 dimensions in 14.0s


### 3.3 Build the vector index

FAISS `IndexFlatIP` means *exact* nearest-neighbour search with dot product - no training, no approximation, nothing to get wrong. At 458 chunks it is instant. The index is saved to `rag_artifacts/chunks.faiss`; searching it later needs only the index file plus `chunks.jsonl` (row order must match).

In [15]:
import faiss

index = faiss.IndexFlatIP(chunk_vectors.shape[1])   # inner product = cosine on normalised vectors
index.add(chunk_vectors)

ARTIFACT_DIR.mkdir(exist_ok=True)
index_path = ARTIFACT_DIR / "chunks.faiss"
faiss.write_index(index, str(index_path))
print(f"index holds {index.ntotal} vectors, saved to {index_path.relative_to(ROOT)}")

index holds 458 vectors, saved to rag_artifacts/chunks.faiss


### 3.4 Index statistics

In [16]:
norms = np.linalg.norm(chunk_vectors, axis=1)
print(f"model           : {EMBEDDING_MODEL}")
print(f"matrix          : {chunk_vectors.shape[0]} vectors x {chunk_vectors.shape[1]} dims")
print(f"index type      : {type(index).__name__} (exact search)")
print(f"file size       : {index_path.stat().st_size / 1024:.1f} KB")
print(f"vector norms    : min {norms.min():.4f}  max {norms.max():.4f}  (all ~1.0 = normalised)")
print(f"chunk row order : matches chunks.jsonl line order")

model           : intfloat/multilingual-e5-small
matrix          : 458 vectors x 384 dims
index type      : IndexFlatIP (exact search)
file size       : 687.0 KB
vector norms    : min 1.0000  max 1.0000  (all ~1.0 = normalised)
chunk row order : matches chunks.jsonl line order


### 3.5 Cross-lingual sanity check

Two English questions and two Swahili questions. Read the results two ways:

- **Scores** — a Swahili question scoring in the same ballpark as English (~0.75+) proves the two languages sit close together in this vector space. That is the signal everything rests on.
- **Rankings** — English rankings must be obviously correct. Swahili rankings are allowed to be weaker *at this stage*: section 4 translates Swahili questions into English **before** retrieval (the translation pivot), and section 7 tunes against measured metrics. This check only asks: does the cross-lingual signal exist at all?

Watch for one pattern: pure number-table chunks from the statistics report score surprisingly well on anything — section 2.7 exists to help you decide whether to drop them from the corpus.

In [17]:
def search(query, k=TOP_K):
    """Return [(score, chunk), ...] for the k chunks closest to the query."""
    query_vector = embed_texts([query], "query")
    scores, row_ids = index.search(query_vector, k)
    return [(float(score), chunks[row]) for score, row in zip(scores[0], row_ids[0])]


TEST_QUERIES = [
    "What fertiliser should I apply to maize?",
    "What are the symptoms of common rust?",
    "Ni mbolea gani inafaa kwa mahindi?",           # What fertiliser is suitable for maize?
    "Magonjwa ya mahindi ni yapi?",                 # What are the diseases of maize?
]

for question in TEST_QUERIES:
    print(f"\nQ: {question}")
    for score, chunk in search(question, 3):
        snippet = chunk["text"][:90].replace("\n", " ")
        print(f"   {score:.3f}  [{chunk['chunk_id']}] {chunk['title'][:40]:42s} {snippet}...")


Q: What fertiliser should I apply to maize?



Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Batches: 100%|██████████| 1/1 [00:00<00:00, 52.28it/s]

   0.896  [S02-p0002-c001] Maize Growers Guide                        2 Soils and climate Maize grows best on deep, well-drained, fertile soils, and where total...
   0.877  [S02-p0004-c001] Maize Growers Guide                        4  The basal fertiliser must be applied before or at the time of planting, preferably cup...
   0.876  [S02-p0017-c001] Maize Growers Guide                        17 Mid-season management 1. Weed control, especially in the first 10 weeks after crop emer...

Q: What are the symptoms of common rust?



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...


Batches: 100%|██████████| 1/1 [00:00<00:00, 79.20it/s]

   0.810  [S03-p0189-c002] National Agriculture Production Report 2   10 14 Kilifi 403 76 391 112 372 42 243 45 26 6 Kirinyaga 93 108 86 71 73 25 110 149 40 61 ...
   0.807  [S03-p0115-c002] National Agriculture Production Report 2   1,255.1 323.9 78.8 34.0 110.0 37.3 29.7 3,859,793.7 1,738,320.0 6,263,852.0 2,388,594.3 2,...
   0.805  [S03-p0186-c002] National Agriculture Production Report 2   29,692 33,534 39,471 Kilifi 71,782 44,585 53,943 26,176 72,748 47,780 95,674 81,813 51,393...

Q: Magonjwa ya mahindi ni yapi?



Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Batches: 100%|██████████| 1/1 [00:00<00:00, 77.55it/s]

   0.784  [S03-p0189-c002] National Agriculture Production Report 2   10 14 Kilifi 403 76 391 112 372 42 243 45 26 6 Kirinyaga 93 108 86 71 73 25 110 149 40 61 ...
   0.782  [S03-p0186-c002] National Agriculture Production Report 2   29,692 33,534 39,471 Kilifi 71,782 44,585 53,943 26,176 72,748 47,780 95,674 81,813 51,393...
   0.777  [S03-p0115-c002] National Agriculture Production Report 2   1,255.1 323.9 78.8 34.0 110.0 37.3 29.7 3,859,793.7 1,738,320.0 6,263,852.0 2,388,594.3 2,...
